# 03 · Model testing
Evaluates both saved models on the untouched hold-out set: ROC-AUC, PR-AUC, Brier score, lift by decile, calibration and SHAP drivers. Saves **artifact 6** (model comparison) and **artifact 7** (charts) and checks the acceptance criteria from the brief.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import shap
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

import pipeline_config as cfg

df = pd.read_csv(cfg.CLEAN_FILE)
split = pd.read_csv(cfg.SPLIT_FILE)
test_df = df[df["customer_id"].isin(split.loc[split["split"] == "test", "customer_id"])]
features = cfg.feature_columns(df)
X_test, y_test = test_df[features], test_df[cfg.TARGET]
models = {p.stem: joblib.load(p) for p in sorted(cfg.MODEL_DIR.glob("*.joblib"))}
print(f"test set: {len(test_df):,} customers, {y_test.mean():.1%} positive | models: {list(models)}")

## 1. Hold-out metrics

In [ ]:
scores = {name: model.predict(X_test) for name, model in models.items()}

rows = []
for name, s in scores.items():
    rows.append({"model": name, "roc_auc": roc_auc_score(y_test, s),
                 "pr_auc": average_precision_score(y_test, s), "brier": brier_score_loss(y_test, s)})
comparison = pd.DataFrame(rows).set_index("model").round(3)
comparison.loc["random (base rate)", "pr_auc"] = round(y_test.mean(), 3)
comparison

## 2. Lift by decile (gradient boosting)

In [ ]:
best = "gradient_boosting"
scored = test_df.assign(score=scores[best])
scored["decile"] = pd.qcut(scored["score"].rank(method="first", ascending=False), 10, labels=range(1, 11))
deciles = scored.groupby("decile", observed=True)[cfg.TARGET].agg(customers="size", conversion_rate="mean")
deciles["lift"] = deciles["conversion_rate"] / y_test.mean()
top20_lift = scored.nlargest(int(len(scored) * 0.2), "score")[cfg.TARGET].mean() / y_test.mean()
print(f"Top-20% lift: {top20_lift:.2f}x")
deciles.round(3)

In [ ]:
cfg.CHART_DIR.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(deciles.index.astype(str), deciles["lift"], color="#2f6db5")
ax.axhline(1, color="grey", linestyle="--", linewidth=1)
ax.set(xlabel="Score decile (1 = highest propensity)", ylabel="Lift vs average",
       title="Conversion lift by decile - hold-out set")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "05_lift_by_decile.png", dpi=150)
plt.show()

## 3. Calibration

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
CalibrationDisplay.from_predictions(y_test, scores[best], n_bins=10, strategy="quantile", ax=ax,
                                    name="gradient boosting")
ax.set_title("Calibration - hold-out set")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "06_calibration.png", dpi=150)
plt.show()
print(f"mean predicted {scores[best].mean():.3f} vs actual {y_test.mean():.3f}")

## 4. Drivers (SHAP)

In [ ]:
pipe = models[best]
preprocess, booster = pipe[:-1], pipe[-1]
sample = X_test.sample(1000, random_state=cfg.RANDOM_STATE)
X_sample = pd.DataFrame(preprocess.transform(sample), columns=preprocess.get_feature_names_out())
shap_values = shap.TreeExplainer(booster).shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, max_display=12, show=False)
plt.title("Top drivers of cross-sell propensity")
plt.tight_layout()
plt.savefig(cfg.CHART_DIR / "07_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, feature in zip(axes, ["tenure_months", "num_products", "email_open_rate"]):
    shap.dependence_plot(feature, shap_values, X_sample, interaction_index=None, ax=ax, show=False)
    ax.axhline(0, color="grey", linewidth=0.8)
fig.suptitle("How each driver moves propensity (SHAP dependence: above 0 = more likely to buy)")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "08_shap_dependence.png", dpi=150)
plt.show()

## 5. Acceptance check and save

In [ ]:
base_pr = comparison.loc["baseline_logreg", "pr_auc"]
checks = pd.DataFrame([
    {"criterion": f"hold-out ROC-AUC >= {cfg.ACCEPTANCE['roc_auc']}", "value": comparison.loc[best, "roc_auc"]},
    {"criterion": f"PR-AUC beats baseline ({base_pr})", "value": comparison.loc[best, "pr_auc"]},
    {"criterion": f"top-20% lift >= {cfg.ACCEPTANCE['top20_lift']}x", "value": round(top20_lift, 2)},
])
checks["pass"] = [checks.loc[0, "value"] >= cfg.ACCEPTANCE["roc_auc"],
                  checks.loc[1, "value"] > base_pr,
                  checks.loc[2, "value"] >= cfg.ACCEPTANCE["top20_lift"]]

comparison.to_csv(cfg.COMPARISON_FILE)
deciles.to_csv(cfg.CHART_DIR / "lift_by_decile.csv")
checks